In [1]:
import pandas as pd
import re
from rdflib import Graph, Namespace, RDF, RDFS, XSD, Literal, URIRef


In [4]:
listings = pd.read_csv("data/listings.csv")
demographics = pd.read_csv("data/demographics.csv")
airbnb_pressure = pd.read_csv("data/airbnb_pressure_by_borough.csv")
combined_pressure = pd.read_csv("data/combined_pressure_by_borough.csv")
housing_pressure = pd.read_csv("data\\housing_pressure_by_borough.csv")

In [ ]:
# Create graph
g = Graph()
EX = Namespace("http://example.org/london-airbnb/")

g.bind("ex", EX)
g.bind("rdf", RDF)
g.bind("rdfs", RDFS)
g.bind("xsd", XSD)

# RDFS classes
classes = [
    "Borough",
    "Listing",
    "Host",
    "RoomType",
    "HousingIndicator",
    "PressureIndicator"
]

for cls in classes:
    g.add((EX[cls], RDF.type, RDFS.Class))

# Object properties
object_properties = {
    "isLocatedIn": ("Listing", "Borough"),
    "hasRoomType": ("Listing", "RoomType"),
    "hasListing": ("Host", "Listing"),
    "hasHousingIndicator": ("Borough", "HousingIndicator"),
    "hasPressureIndicator": ("Borough", "PressureIndicator"),
}

for prop, (domain, range_) in object_properties.items():
    g.add((EX[prop], RDF.type, RDF.Property))
    g.add((EX[prop], RDFS.domain, EX[domain]))
    g.add((EX[prop], RDFS.range, EX[range_]))

# Datatype properties
datatype_properties = {
    "listingID": ("Listing", XSD.integer),
    "listingName": ("Listing", XSD.string),
    "priceNight": ("Listing", XSD.float),
    "availability": ("Listing", XSD.integer),
    "reviewsMonth": ("Listing", XSD.float),
    "latitude": ("Listing", XSD.float),
    "longitude": ("Listing", XSD.float),
    "minimumNights": ("Listing", XSD.integer),
    "numberOfReviews": ("Listing", XSD.integer),

    "hostID": ("Host", XSD.integer),
    "hostName": ("Host", XSD.string),
    "calculatedHostListingsCount": ("Host", XSD.integer),

    "boroughName": ("Borough", XSD.string),
    "populationEstimate": ("Borough", XSD.integer),
    "householdEstimate": ("Borough", XSD.integer),
    "populationDensity": ("Borough", XSD.float),
    "medianHousePrice": ("Borough", XSD.float),
    "medianIncome": ("Borough", XSD.float),
    "newHomes": ("Borough", XSD.integer),
    "ownedRatio": ("Borough", XSD.float),
    "rentedAssociationRatio": ("Borough", XSD.float),
    "rentedPrivateRatio": ("Borough", XSD.float),
    "transportAccessibility": ("Borough", XSD.float),

    "roomTypeName": ("RoomType", XSD.string),

    "airbnbPressureScore": ("PressureIndicator", XSD.float),
    "airbnbPressureLevel": ("PressureIndicator", XSD.string),
    "housingPressureScore": ("HousingIndicator", XSD.float),
    "housingPressureLevel": ("HousingIndicator", XSD.string),
    "derivedFrom": ("PressureIndicator", XSD.string),
}

for prop, (domain, range_) in datatype_properties.items():
    g.add((EX[prop], RDF.type, RDF.Property))
    g.add((EX[prop], RDFS.domain, EX[domain]))
    g.add((EX[prop], RDFS.range, range_))

# CSV column to RDF property mappings
listing_literal_mapping = {
    "id": (EX.listingID, XSD.integer),
    "name": (EX.listingName, XSD.string),
    "price": (EX.priceNight, XSD.float),
    "availability_365": (EX.availability, XSD.integer),
    "reviews_per_month": (EX.reviewsMonth, XSD.float),
    "latitude": (EX.latitude, XSD.float),
    "longitude": (EX.longitude, XSD.float),
    "minimum_nights": (EX.minimumNights, XSD.integer),
    "number_of_reviews": (EX.numberOfReviews, XSD.integer),
}

host_literal_mapping = {
    "host_id": (EX.hostID, XSD.integer),
    "host_name": (EX.hostName, XSD.string),
    "calculated_host_listings_count": (EX.calculatedHostListingsCount, XSD.integer),
}

demo_literal_mapping = {
    "Population Estimate 2016": (EX.populationEstimate, XSD.integer),
    "Household Estimate 2016": (EX.householdEstimate, XSD.integer),
    "Population density 2016": (EX.populationDensity, XSD.float),
    "Median House Price 2014": (EX.medianHousePrice, XSD.float),
    "Median Household Income estimate 2012/13": (EX.medianIncome, XSD.float),
    "Net new homes 2014-2015": (EX.newHomes, XSD.integer),
    "% of households owned 2014": (EX.ownedRatio, XSD.float),
    "% of households social rented 2014": (EX.rentedAssociationRatio, XSD.float),
    "% of households private rented 2014": (EX.rentedPrivateRatio, XSD.float),
    "Average Public Transport Accessibility score 2014": (EX.transportAccessibility, XSD.float),
}

# Convert listings.csv to RDF
for _, row in listings.iterrows():

    listing = EX[f"listing/{row['id']}"]
    host = EX[f"host/{row['host_id']}"]
    borough = EX[f"borough/{row['neighbourhood'].replace(' ', '_')}"]
    room_type = EX[f"roomType/{row['room_type'].replace(' ', '_').replace('/', '_')}"]

    g.add((listing, RDF.type, EX.Listing))
    g.add((host, RDF.type, EX.Host))
    g.add((borough, RDF.type, EX.Borough))
    g.add((room_type, RDF.type, EX.RoomType))

    g.add((host, EX.hasListing, listing))
    g.add((listing, EX.isLocatedIn, borough))
    g.add((listing, EX.hasRoomType, room_type))

    for csv_col, (rdf_prop, datatype) in listing_literal_mapping.items():
        g.add((listing, rdf_prop, Literal(row[csv_col], datatype=datatype)))

    for csv_col, (rdf_prop, datatype) in host_literal_mapping.items():
        g.add((host, rdf_prop, Literal(row[csv_col], datatype=datatype)))

    g.add((borough, EX.boroughName, Literal(row["neighbourhood"], datatype=XSD.string)))
    g.add((room_type, EX.roomTypeName, Literal(row["room_type"], datatype=XSD.string)))

# Convert demographics.csv to RDF
borough_column = "Area name"

for _, row in demographics.iterrows():

    borough = EX[f"borough/{row[borough_column].replace(' ', '_')}"]

    g.add((borough, RDF.type, EX.Borough))
    g.add((borough, EX.boroughName, Literal(row[borough_column], datatype=XSD.string)))

    for csv_col, (rdf_prop, datatype) in demo_literal_mapping.items():
        g.add((borough, rdf_prop, Literal(row[csv_col], datatype=datatype)))

# Derived Airbnb pressure indicator
borough_stats = listings.groupby("neighbourhood").agg(
    listing_count=("id", "count")
).reset_index()

min_count = borough_stats["listing_count"].min()
max_count = borough_stats["listing_count"].max()

def pressure_level(score):
    if score < 0.33:
        return "Low"
    elif score < 0.66:
        return "Medium"
    else:
        return "High"

for _, row in borough_stats.iterrows():

    borough = EX[f"borough/{row['neighbourhood'].replace(' ', '_')}"]
    indicator = EX[f"pressureIndicator/{row['neighbourhood'].replace(' ', '_')}"]

    score = (row["listing_count"] - min_count) / (max_count - min_count)

    g.add((indicator, RDF.type, EX.PressureIndicator))
    g.add((borough, EX.hasPressureIndicator, indicator))

    g.add((indicator, EX.airbnbPressureScore, Literal(round(score, 3), datatype=XSD.float)))
    g.add((indicator, EX.airbnbPressureLevel, Literal(pressure_level(score), datatype=XSD.string)))
    g.add((indicator, EX.derivedFrom, Literal("listing_count", datatype=XSD.string)))

# Save RDF files
g.serialize(destination="london_airbnb_kg.ttl", format="turtle")

print("RDF graph created successfully.")
print("Total triples:", len(g))
print("Saved files:")
print("- london_airbnb_kg.ttl")
print("- london_airbnb_kg.rdf")

#SPARQL query Test
query = """
PREFIX ex: <http://example.org/london-airbnb/>

SELECT ?borough ?level ?score
WHERE {
    ?borough ex:hasPressureIndicator ?indicator .
    ?indicator ex:airbnbPressureLevel ?level .
    ?indicator ex:airbnbPressureScore ?score .
}
LIMIT 10
"""

print("\nTest query results:")
for result in g.query(query):
    print(result)